In [1]:
import cv2
import mediapipe as mp
import math
import time

# Start Webcam
cap = cv2.VideoCapture(0)

# Speed Optimization
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

# MediaPipe Face Mesh
mp_face_mesh = mp.solutions.face_mesh

face_mesh = mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# Blink Variables
blink_detected = False
snap_time = 0

while True:

    # Read Camera Frame
    success, img = cap.read()

    if not success:
        break

    # Mirror Effect
    img = cv2.flip(img, 1)

    h, w, _ = img.shape

    # Convert BGR to RGB
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Process Face Mesh
    results = face_mesh.process(rgb)

    # If Face Detected
    if results.multi_face_landmarks:

        face = results.multi_face_landmarks[0]

        # LEFT EYE LANDMARKS
        top = face.landmark[159]
        bottom = face.landmark[145]
        left = face.landmark[33]
        right = face.landmark[133]

        # Convert Coordinates
        tx, ty = int(top.x * w), int(top.y * h)
        bx, by = int(bottom.x * w), int(bottom.y * h)

        lx, ly = int(left.x * w), int(left.y * h)
        rx, ry = int(right.x * w), int(right.y * h)

        # Draw Eye Tracking Lines
        cv2.line(img, (tx, ty), (bx, by), (0,255,255), 3)
        cv2.line(img, (lx, ly), (rx, ry), (255,0,255), 3)

        # Eye Measurements
        vertical = math.hypot(tx - bx, ty - by)
        horizontal = math.hypot(lx - rx, ly - ry)

        # Eye Ratio
        ratio = vertical / horizontal

        # Blink Detection
        if ratio < 0.18:

            if not blink_detected:

                blink_detected = True

                # Save Image to Pictures Folder
                filename = f"C:/Users/user/Pictures/Snap_{int(time.time()*1000)}.jpg"

                # Save Photo
                cv2.imwrite(filename, img)

                # Terminal Feedback
                print("PHOTO SAVED!")

                # Snap Timer
                snap_time = time.time()

        else:
            blink_detected = False

    # Show Snap Message
    if time.time() - snap_time < 0.6:

        cv2.putText(
            img,
            "SNAPPED! 📸",
            (140,100),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.5,
            (0,255,0),
            4
        )

    else:

        cv2.putText(
            img,
            "BLINK TO SNAP 👀",
            (100,50),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (255,255,0),
            3
        )

    # Main Title
    cv2.putText(
        img,
        "PPSL AI SNAP CAMERA",
        (60,450),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0,255,255),
        2
    )

    # Show Camera
    cv2.imshow("PPSL AI SNAP CAMERA", img)

    # Quit
    if cv2.waitKey(1) & 0xFF == ord('f'):
        break

# Release Camera
cap.release()

# Close Windows
cv2.destroyAllWindows()

PHOTO SAVED!
